# 黄金价格 vs 战争事件可视化分析

本 Notebook 分析历次战争期间黄金价格的表现，并评估跨国套利空间。

---

## 数据源

### 黄金价格数据

| 方式 | 数据范围 | 说明 |
|------|---------|------|
| **Investing.com** (推荐) | 1970至今 | [下载链接](https://www.investing.com/commodities/gold-historical-data)，保存为 `gold_price_history.csv` |
| **akshare** | 2016至今 | 自动获取上海黄金交易所数据 |

### 汇率数据

- 通过 `akshare` 自动获取 USD/CNY 汇率历史数据

---

In [ ]:
# 安装依赖（如需要，取消注释）
# !pip install akshare plotly pandas numpy -q

In [ ]:
# ============================================
# 导入所有依赖库
# ============================================

import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import os
import warnings
import akshare as ak

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✓ 所有库加载完成")

---
## Part 1: 数据加载

In [ ]:
# ============================================
# 1. 加载战争事件数据
# ============================================

with open('war_events.json', 'r', encoding='utf-8') as f:
    wars = json.load(f)

print("✓ 已加载战争事件数据：")
for war_id, war_info in wars.items():
    end_date = war_info['end'] if war_info['end'] else '至今'
    print(f"  • {war_info['name']}: {war_info['start']} ~ {end_date}")

In [ ]:
# ============================================
# 2. 加载黄金价格数据
# ============================================

def load_gold_from_csv(filepath):
    """从 CSV 文件加载（Investing.com 格式）"""
    df = pd.read_csv(filepath)
    
    # Investing.com 格式: Date, Price, Open, High, Low, Vol., Change%
    col_map = {'Date': 'date', 'Price': 'close', 'Open': 'open', 'High': 'high', 'Low': 'low'}
    df = df.rename(columns=col_map)
    
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    for col in ['close', 'open', 'high', 'low']:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].astype(str).str.replace(',', '').astype(float)
    
    return df[['date', 'close']].dropna()


def load_gold_from_akshare():
    """从 akshare 获取上海黄金交易所数据"""
    print("正在从 akshare 获取黄金数据...")
    df = ak.spot_hist_sge(symbol='Au99.99')
    df = df[['date', 'close']].copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    return df


# 检查数据源
csv_file = 'gold_price_history.csv'

if os.path.exists(csv_file):
    print(f"✓ 发现本地数据文件: {csv_file}")
    df_gold = load_gold_from_csv(csv_file)
else:
    print("⚠ 未找到 gold_price_history.csv，使用 akshare 获取数据（仅覆盖 2016 年至今）")
    df_gold = load_gold_from_akshare()

print(f"\n✓ 黄金数据加载完成")
print(f"  时间范围: {df_gold['date'].min().strftime('%Y-%m-%d')} ~ {df_gold['date'].max().strftime('%Y-%m-%d')}")
print(f"  数据行数: {len(df_gold):,}")

In [ ]:
# ============================================
# 3. 加载汇率数据并合并
# ============================================

print("正在获取 USD/CNY 汇率数据...")

try:
    fx_df = ak.currency_boc_safe()
    
    # 处理列名
    date_col = None
    price_col = None
    
    for col in fx_df.columns:
        if 'date' in col.lower() or '日期' in col:
            date_col = col
        if 'usd' in col.lower() or 'cny' in col.lower() or '汇率' in col or 'price' in col.lower():
            price_col = col
    
    if date_col is None:
        date_col = fx_df.columns[0]
    if price_col is None:
        price_col = fx_df.columns[1]
    
    fx_df['date'] = pd.to_datetime(fx_df[date_col])
    fx_df = fx_df.sort_values('date').reset_index(drop=True)
    fx_df['usdcny'] = fx_df[price_col].astype(float)
    fx_df = fx_df[['date', 'usdcny']].dropna()
    
    print(f"✓ 汇率数据加载完成")
    print(f"  时间范围: {fx_df['date'].min().strftime('%Y-%m-%d')} ~ {fx_df['date'].max().strftime('%Y-%m-%d')}")
    
    # 合并黄金和汇率数据
    df_merged = pd.merge_asof(
        df_gold.sort_values('date'),
        fx_df[['date', 'usdcny']].sort_values('date'),
        on='date',
        direction='nearest',
        tolerance=pd.Timedelta('3 days')
    )
    df_merged = df_merged.dropna(subset=['usdcny'])
    
    print(f"\n✓ 数据合并完成，合并后数据点: {len(df_merged):,}")
    HAS_FX_DATA = True
    
except Exception as e:
    print(f"⚠ 汇率数据获取失败: {e}")
    print("  → 套利分析将被跳过")
    df_merged = df_gold.copy()
    df_merged['usdcny'] = np.nan
    HAS_FX_DATA = False

---
## Part 2: 分析函数定义

In [ ]:
# ============================================
# 黄金价格分析函数
# ============================================

def analyze_war(df, war_id, wars, days_before=90, days_after=180):
    """分析单个战争期间的黄金价格"""
    war = wars[war_id]
    start_date = pd.to_datetime(war['start'])
    end_date = pd.to_datetime(war['end']) if war['end'] else df['date'].max()
    
    window_start = start_date - timedelta(days=days_before)
    window_end = end_date + timedelta(days=days_after) if war['end'] else df['date'].max()
    
    mask = (df['date'] >= window_start) & (df['date'] <= window_end)
    war_df = df[mask].copy()
    
    if war_df.empty or len(war_df) < 5:
        return None
    
    pre_mask = war_df['date'] < start_date
    base_price = war_df[pre_mask]['close'].iloc[-1] if pre_mask.any() else war_df['close'].iloc[0]
    
    war_df['pct_change'] = (war_df['close'] / base_price - 1) * 100
    
    during_mask = (war_df['date'] >= start_date) & (war_df['date'] <= end_date)
    during = war_df[during_mask]
    
    if during.empty:
        return None
    
    stats = {
        'war_name': war['name'],
        'name_en': war['name_en'],
        'color': war['color'],
        'base_price': base_price,
        'max_price': during['close'].max(),
        'min_price': during['close'].min(),
        'max_change': during['pct_change'].max(),
        'min_change': during['pct_change'].min(),
        'end_price': war_df[war_df['date'] <= end_date]['close'].iloc[-1],
        'end_change': war_df[war_df['date'] <= end_date]['pct_change'].iloc[-1],
    }
    
    return {'data': war_df, 'stats': stats, 'war_info': war, 'window': (window_start, window_end, start_date, end_date)}


def plot_war(result, title=None):
    """绘制单个战争期间价格图"""
    if result is None:
        return None
    
    df = result['data']
    war = result['war_info']
    _, _, war_start, war_end = result['window']
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df['date'], y=df['close'],
        mode='lines', name='黄金价格',
        line=dict(color='#FFD700', width=2),
        hovertemplate='日期: %{x|%Y-%m-%d}<br>价格: $%{y:.2f}<extra></extra>'
    ))
    
    fig.add_vrect(
        x0=war_start, x1=war_end if war['end'] else df['date'].max(),
        fillcolor=war['color'], opacity=0.2, layer='below', line_width=0
    )
    
    for event in war['events']:
        event_date = pd.to_datetime(event['date'])
        if df['date'].min() <= event_date <= df['date'].max():
            idx = min(df['date'].searchsorted(event_date), len(df) - 1)
            price = df.iloc[idx]['close']
            colors = {'start': '#e74c3c', 'end': '#27ae60', 'military': '#f39c12', 'political': '#3498db'}
            
            fig.add_trace(go.Scatter(
                x=[event_date], y=[price],
                mode='markers', name=event['label'],
                marker=dict(size=10, color=colors.get(event['type'], '#95a5a6'), symbol='triangle-down'),
                hovertemplate=f'{event["label"]}<extra></extra>'
            ))
    
    fig.add_vline(x=war_start, line_dash='dash', line_color='red', opacity=0.7)
    
    fig.update_layout(
        title=title or f"{war['name']}期间黄金价格走势",
        xaxis_title='日期', yaxis_title='价格 (美元/盎司)',
        template='plotly_white', height=500, hovermode='x unified',
        legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
    )
    
    return fig


def plot_comparison(df, wars, days_before=30, days_after=90):
    """多战争对比图（归一化）"""
    fig = go.Figure()
    
    for war_id, war in wars.items():
        start = pd.to_datetime(war['start'])
        end = pd.to_datetime(war['end']) if war['end'] else df['date'].max()
        
        window_start = start - timedelta(days=days_before)
        window_end = end + timedelta(days=days_after) if war['end'] else df['date'].max()
        
        mask = (df['date'] >= window_start) & (df['date'] <= window_end)
        war_df = df[mask].copy()
        
        if len(war_df) < 5:
            continue
        
        idx = min(war_df['date'].searchsorted(start), len(war_df) - 1)
        base = war_df.iloc[idx]['close']
        
        if pd.isna(base) or base == 0:
            continue
        
        war_df['normalized'] = (war_df['close'] / base) * 100
        war_df['days'] = (war_df['date'] - start).dt.days
        
        fig.add_trace(go.Scatter(
            x=war_df['days'], y=war_df['normalized'],
            mode='lines', name=war['name_en'],
            line=dict(color=war['color'], width=2)
        ))
    
    fig.add_vline(x=0, line_dash='dash', line_color='black', opacity=0.5)
    fig.add_hline(y=100, line_dash='dot', line_color='gray', opacity=0.5)
    
    fig.update_layout(
        title='历次战争期间黄金价格变化对比（战争开始日=100）',
        xaxis_title='距战争开始天数', yaxis_title='相对价格指数',
        template='plotly_white', height=600,
        legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
    )
    
    return fig

In [ ]:
# ============================================
# 套利空间分析函数
# ============================================

def calculate_arbitrage(merged_df, wars_data):
    """
    计算战争期间的套利空间
    
    套利逻辑：
    - 汇率波动会产生跨境套利机会
    - 人民币升值(USDCNY下降)会使国内黄金相对更贵
    """
    results = []
    
    for war_id, war in wars_data.items():
        war_start = pd.to_datetime(war['start'])
        war_end = pd.to_datetime(war['end']) if war['end'] else merged_df['date'].max()
        
        war_mask = (merged_df['date'] >= war_start) & (merged_df['date'] <= war_end)
        war_data = merged_df[war_mask].copy()
        
        if len(war_data) < 3 or war_data['usdcny'].isna().all():
            continue
        
        # 汇率指标
        fx_start = war_data['usdcny'].dropna().iloc[0]
        fx_end = war_data['usdcny'].dropna().iloc[-1]
        fx_min = war_data['usdcny'].min()
        fx_max = war_data['usdcny'].max()
        
        # 黄金指标
        gold_start = war_data['close'].iloc[0]
        gold_max = war_data['close'].max()
        gold_return = (gold_max / gold_start - 1) * 100
        
        # 套利空间
        fx_change_pct = (fx_end / fx_start - 1) * 100
        fx_range_pct = (fx_max / fx_min - 1) * 100
        fx_volatility = war_data['usdcny'].std() / war_data['usdcny'].mean() * 100
        estimated_arbitrage = gold_return + fx_range_pct * 0.3
        
        results.append({
            'war_name': war['name'],
            'war_id': war_id,
            'gold_return_pct': gold_return,
            'fx_start': fx_start,
            'fx_end': fx_end,
            'fx_change_pct': fx_change_pct,
            'fx_range_pct': fx_range_pct,
            'fx_volatility': fx_volatility,
            'estimated_max_arbitrage': estimated_arbitrage,
            'arbitrage_efficiency': gold_return / fx_range_pct if fx_range_pct > 0 else float('inf'),
            'color': war['color']
        })
    
    return results


def plot_arbitrage(arb_results):
    """套利空间可视化"""
    if not arb_results:
        return None
    
    arb_df = pd.DataFrame(arb_results)
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            '黄金涨幅 vs 汇率波动',
            '估计最大套利空间',
            '套利效率比',
            '汇率波动幅度'
        ),
        specs=[[{"secondary_y": True}, {"type": "bar"}],
               [{"type": "bar"}, {"type": "bar"}]],
        vertical_spacing=0.15
    )
    
    wars_list = list(arb_df['war_name'])
    colors = list(arb_df['color'])
    
    # 图1: 黄金涨幅 vs 汇率波动
    fig.add_trace(go.Bar(x=wars_list, y=arb_df['gold_return_pct'], name='黄金涨幅 (%)', 
                         marker_color='#FFD700', opacity=0.8), row=1, col=1, secondary_y=False)
    fig.add_trace(go.Bar(x=wars_list, y=arb_df['fx_range_pct'], name='汇率波动 (%)',
                         marker_color='#3498db', opacity=0.8), row=1, col=1, secondary_y=True)
    
    # 图2: 估计最大套利空间
    fig.add_trace(go.Bar(x=wars_list, y=arb_df['estimated_max_arbitrage'], 
                         marker_color=colors, showlegend=False), row=1, col=2)
    
    # 图3: 套利效率比
    efficiency = arb_df['arbitrage_efficiency'].replace([float('inf')], None)
    fig.add_trace(go.Bar(x=wars_list, y=efficiency, marker_color=colors, showlegend=False), row=2, col=1)
    
    # 图4: 汇率波动幅度
    fig.add_trace(go.Bar(x=wars_list, y=arb_df['fx_range_pct'], 
                         marker_color='#e74c3c', showlegend=False), row=2, col=2)
    
    fig.update_layout(
        title_text='跨国黄金套利空间分析（USD/CNY 汇率波动影响）',
        height=700, template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    fig.update_xaxes(tickangle=45)
    
    return fig


print("✓ 分析函数定义完成")

---
## Part 3: 数据概览

In [ ]:
# ============================================
# 数据覆盖检查
# ============================================

print("=" * 60)
print("数据覆盖情况")
print("=" * 60)

data_start = df_gold['date'].min()
data_end = df_gold['date'].max()
available_wars = []

for war_id, war in wars.items():
    war_start = pd.to_datetime(war['start'])
    war_end = pd.to_datetime(war['end']) if war['end'] else data_end
    
    if war_end >= data_start and war_start <= data_end:
        overlap_start = max(war_start, data_start)
        overlap_end = min(war_end, data_end)
        
        if war['end']:
            coverage = (overlap_end - overlap_start).days / (war_end - war_start).days * 100
        else:
            coverage = 100
        
        status = "✓" if coverage > 10 else "⚠"
        print(f"{status} {war['name']}: {coverage:.0f}% 覆盖")
        available_wars.append(war_id)
    else:
        print(f"✗ {war['name']}: 无数据")

print(f"\n可分析战争数: {len(available_wars)} / {len(wars)}")

---
## Part 4: 黄金价格分析

In [ ]:
# ============================================
# 单战争详细分析（以俄乌战争为例）
# ============================================

result = analyze_war(df_gold, 'russo_ukrainian_war', wars, days_before=180, days_after=365)

if result:
    print(f"\n{'='*50}")
    print(f"{result['war_info']['name']} 黄金价格分析")
    print(f"{'='*50}")
    for k, v in result['stats'].items():
        if isinstance(v, float):
            print(f"  {k}: {v:.2f}")
    
    fig = plot_war(result)
    if fig:
        fig.show()
else:
    print("⚠ 数据不足，无法分析俄乌战争")

In [ ]:
# ============================================
# 多战争对比（归一化）
# ============================================

fig = plot_comparison(df_gold, wars, days_before=30, days_after=180)
if fig:
    fig.show()

In [ ]:
# ============================================
# 统计汇总
# ============================================

summary = []
for war_id in wars.keys():
    result = analyze_war(df_gold, war_id, wars, days_before=30, days_after=90)
    if result and result['stats']:
        summary.append(result['stats'])

if summary:
    summary_df = pd.DataFrame(summary).set_index('war_name')
    
    print("\n黄金价格统计汇总表：")
    display(summary_df[['base_price', 'max_price', 'max_change', 'end_price', 'end_change']].round(2))
    
    # 可视化
    fig = make_subplots(rows=1, cols=2, subplot_titles=('最大涨幅 (%)', '结束时涨跌幅 (%)'))
    
    names = list(summary_df.index)
    colors = [s['color'] for s in summary]
    
    fig.add_trace(go.Bar(x=names, y=summary_df['max_change'], marker_color=colors), row=1, col=1)
    fig.add_trace(go.Bar(x=names, y=summary_df['end_change'], marker_color=colors), row=1, col=2)
    
    fig.update_layout(
        title_text='战争期间黄金表现汇总', 
        showlegend=False, height=400, template='plotly_white'
    )
    fig.update_xaxes(tickangle=45)
    fig.show()
else:
    print("⚠ 没有足够的战争数据进行汇总")

---
## Part 5: 跨国套利空间分析

**分析逻辑**：
- 跨境套利利用两地市场的黄金价差 + 汇率波动实现超额收益
- 汇率波动幅度直接影响套利空间
- 假设条件：交易成本 0.8%（双边 0.5% + 汇率转换 0.3%）

**核心指标说明**：

| 指标 | 公式 | 含义 |
|------|------|------|
| **黄金涨幅** | (最高价 / 起始价 - 1) × 100% | 战争期间黄金最大涨幅 |
| **汇率波动** | (最高汇率 / 最低汇率 - 1) × 100% | USD/CNY 波动幅度 |
| **估计最大套利** | 黄金涨幅 + 汇率波动 × 0.3 | 保守估计可捕获的收益 |
| **套利效率比** | 黄金涨幅 / 汇率波动 | 每单位汇率波动带来的黄金收益 |

**套利效率比解读**：
- **> 10**：高效率 - 黄金涨幅远超汇率波动，套利空间大
- **3 ~ 10**：中等效率 - 黄金涨幅适中，需关注汇率风险
- **< 3**：低效率 - 汇率波动较大，套利收益被汇率风险稀释

In [ ]:
# ============================================
# 套利空间计算
# ============================================

if HAS_FX_DATA:
    arbitrage_results = calculate_arbitrage(df_merged, wars)
    
    if arbitrage_results:
        print(f"\n{'='*70}")
        print("套利空间分析结果")
        print(f"{'='*70}")
        
        for r in arbitrage_results:
            print(f"\n📍 {r['war_name']}")
            print(f"   黄金涨幅: {r['gold_return_pct']:.1f}%")
            print(f"   汇率波动: {r['fx_range_pct']:.2f}% ({r['fx_start']:.3f} → {r['fx_end']:.3f})")
            print(f"   估计最大套利: {r['estimated_max_arbitrage']:.1f}%")
    else:
        print("⚠ 没有足够的套利分析数据")
else:
    print("⚠ 汇率数据不可用，跳过套利分析")
    arbitrage_results = []

In [ ]:
# ============================================
# 套利空间可视化
# ============================================

def get_efficiency_level(ratio):
    """根据套利效率比返回效率等级"""
    if pd.isna(ratio) or ratio == float('inf'):
        return 'N/A'
    elif ratio >= 10:
        return '🟢 高'
    elif ratio >= 3:
        return '🟡 中'
    else:
        return '🔴 低'

if arbitrage_results:
    fig = plot_arbitrage(arbitrage_results)
    if fig:
        fig.show()
    
    # 汇总表
    arb_df = pd.DataFrame(arbitrage_results)
    display_df = arb_df[['war_name', 'gold_return_pct', 'fx_range_pct', 
                         'estimated_max_arbitrage', 'arbitrage_efficiency']].copy()
    display_df.columns = ['战争名称', '黄金涨幅(%)', '汇率波动(%)', '估计最大套利(%)', '套利效率比']
    
    # 添加效率等级列
    display_df['效率等级'] = display_df['套利效率比'].apply(get_efficiency_level)
    
    print("\n套利空间汇总表：")
    
    # 美化显示
    styled_df = display_df.style.format({
        '黄金涨幅(%)': '{:.1f}',
        '汇率波动(%)': '{:.2f}',
        '估计最大套利(%)': '{:.1f}',
        '套利效率比': '{:.1f}'
    })
    
    display(styled_df)
    
    # 效率统计
    high_count = sum(1 for r in arbitrage_results if r['arbitrage_efficiency'] >= 10)
    med_count = sum(1 for r in arbitrage_results if 3 <= r['arbitrage_efficiency'] < 10)
    low_count = sum(1 for r in arbitrage_results if r['arbitrage_efficiency'] < 3)
    
    print(f"\n📈 效率分布：🟢 高效率 {high_count} | 🟡 中效率 {med_count} | 🔴 低效率 {low_count}")

In [ ]:
# ============================================
# 真实收益模拟（扣除成本）
# ============================================

if arbitrage_results:
    TOTAL_COST = 0.8  # 总交易成本 %
    
    # 构建收益数据
    profit_data = []
    for r in arbitrage_results:
        gold_return = r['gold_return_pct']
        fx_contrib = r['fx_range_pct'] * 0.3
        gross = gold_return + fx_contrib
        net = gross - TOTAL_COST
        
        profit_data.append({
            '战争名称': r['war_name'],
            '黄金涨幅(%)': gold_return,
            '汇率贡献(%)': fx_contrib,
            '毛收益(%)': gross,
            '净收益(%)': net
        })
    
    profit_df = pd.DataFrame(profit_data)
    
    print(f"\n{'='*70}")
    print(f"真实套利收益模拟（扣除 {TOTAL_COST}% 交易成本）")
    print(f"{'='*70}\n")
    
    # 使用 DataFrame 样式美化输出
    styled_df = profit_df.style.format({
        '黄金涨幅(%)': '{:.1f}',
        '汇率贡献(%)': '{:.1f}',
        '毛收益(%)': '{:.1f}',
        '净收益(%)': '{:.1f}'
    }).background_gradient(subset=['净收益(%)'], cmap='RdYlGn')
    
    display(styled_df)
    
    # 汇总统计
    avg_net = profit_df['净收益(%)'].mean()
    max_net = profit_df['净收益(%)'].max()
    best_war = profit_df.loc[profit_df['净收益(%)'].idxmax(), '战争名称']
    
    print(f"\n{'='*70}")
    print("📊 收益汇总")
    print(f"{'='*70}")
    print(f"  • 平均净收益: {avg_net:.1f}%")
    print(f"  • 最高净收益: {max_net:.1f}% ({best_war})")
    print(f"  • 交易成本: {TOTAL_COST}%")

---
## Part 6: 综合结论

In [ ]:
# ============================================
# 综合结论输出
# ============================================

print("=" * 70)
print("📊 黄金 vs 战争：综合分析结论")
print("=" * 70)

# 黄金价格结论
if summary:
    print(f"\n【黄金价格表现】")
    print(f"  • 分析战争数: {len(summary)}")
    print(f"  • 平均最大涨幅: {summary_df['max_change'].mean():.1f}%")
    print(f"  • 平均结束时涨幅: {summary_df['end_change'].mean():.1f}%")
    
    best = summary_df['max_change'].idxmax()
    print(f"  • 最大涨幅战争: {best} ({summary_df.loc[best, 'max_change']:.1f}%)")

# 套利空间结论
if arbitrage_results:
    print(f"\n【套利空间分析】")
    print(f"  • 可分析战争数: {len(arbitrage_results)}")
    avg_arb = sum(r['estimated_max_arbitrage'] for r in arbitrage_results) / len(arbitrage_results)
    print(f"  • 平均最大套利空间: {avg_arb:.1f}%")
    print(f"  • 扣除成本后净收益: {avg_arb - 0.8:.1f}%")

print(f"\n{'='*70}")
print("分析完成")
print("=" * 70)